<div class="alert alert-info">

## <center> CS587 Final Project – Phase 1 (Waterfall)</center>
### <center> GenAI-Assisted Project Planning for a Personalized Learning Management System (LMS)</center>

**Framework:** Ollama + Llama 3  
**Experiment Owner:** Anu Singh  
**Purpose:** Simulate Waterfall SDLC planning using Llama 3 as the LLM for multiple software engineering roles.

</div>

In [1]:
import subprocess
import sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "ollama", "--quiet"
])

print("Dependencies installed successfully.")

Dependencies installed successfully.


You should consider upgrading via the '/Users/anusingh/Documents/virtual team lms/venv/bin/python -m pip install --upgrade pip' command.


In [2]:
# -----------------------------
import os
import time
import shutil
from datetime import datetime
import ollama
import json
import requests



In [3]:
env_path = os.path.join(os.getcwd(), ".env")

if os.path.exists(env_path):
    with open(env_path) as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                key, value = line.split("=", 1)
                os.environ[key.strip()] = value.strip()

print("Environment loaded (if .env exists).")

Environment loaded (if .env exists).


In [4]:
FRAMEWORK = "Ollama"
MODEL = "llama3"
TEMPERATURE = 0.1
NUM_PREDICT = 400

print(f"Framework: {FRAMEWORK}")
print(f"Model: {MODEL}")
print("Make sure Ollama is running and the model is pulled.")


Framework: Ollama
Model: llama3
Make sure Ollama is running and the model is pulled.


In [5]:
YELLOW = "\033[33m"
GREEN = "\033[32m"
CYAN = "\033[36m"
RED = "\033[31m"
RESET = "\033[0m"

SEPARATOR = "=" * 100


def print_autogen_style(sender, receiver, message):
    print(f"\n{SEPARATOR}")
    print(f"{YELLOW}{sender}{RESET} -> {GREEN}{receiver}{RESET}")
    print(SEPARATOR)
    print(message[:3000] if isinstance(message, str) else str(message))
    print()


def print_agent_response(agent_name, response_text):
    print(f"\n{CYAN}{agent_name} RESPONSE{RESET}")
    print(SEPARATOR)
    print(response_text[:5000])
    print()

In [13]:
AGENT_SYSTEM_PROMPTS = {
    "Requirement_Engineer": """
You are a Requirements Engineer.

Create:
- Use cases
- Functional requirements
- Non-functional requirements

End with:

## Effort Calculation
Work = total number of requirements
Productivity = 5 requirements per day
Effort = Work / Productivity
""",

    "System_Engineer": """
You are a System Engineer.

Create:
- System architecture
- Component diagram
- Sequence diagram
- ER diagram
- Technology stack

End with:

## Effort Calculation
Work = estimated design pages
Productivity = 5 pages/day
Effort = Work / Productivity
""",

    "Software_Engineer": """
You are a Software Engineer.

Create:
- Implementation task list
- Dependencies
- Development phases

End with:

## 4. Effort Calculation

SLOC = 15000  
Productivity = 50 SLOC/day  

Effort = 15000 / 50 = 300 days
""",

    "Test_Engineer": """
You are a Test Engineer.

Create:
- Test strategy
- Functional tests
- Non-functional tests
- Schedule

End with effort calculation based on
2 test cases per day.
""",

    "Documentation_Engineer": """
You are a Documentation Engineer.

Create:
- User documentation outline
- Developer documentation outline
- Documentation schedule

End with effort calculation based on
3 pages per day.
"""
}

In [14]:
customer_message = (
    "I want to build a GenAI-Assisted Personalized Learning Management System (LMS) "
    "with the following core objectives and capabilities:\n\n"
    "1. Personalized Learning Paths for Students: Use GenAI to analyze a student's "
    "goals, current knowledge, performance, and preferences. Automatically generate "
    "and continuously adapt personalized learning paths (modules, lessons, assessments, "
    "and projects). Recommend content, practice questions, and projects tailored to "
    "each student.\n\n"
    "2. AI Tutor for Students: Provide an AI tutor chatbot inside the LMS that can "
    "answer course-related questions in natural language, explain concepts at different "
    "difficulty levels (beginner to advanced), generate examples, quizzes, and hints, "
    "and suggest next steps or remedial content when students are stuck.\n\n"
    "3. AI Assistance for Instructors: Help instructors create course materials "
    "including lecture outlines, slide drafts, reading summaries, quiz questions, "
    "assignments, rubrics, and sample answers. Allow instructors to input a topic or "
    "syllabus and have the system propose structured modules, learning objectives, "
    "and suggested assessments.\n\n"
    "4. Progress Tracking Dashboards: Dashboards for students showing learning path "
    "progress, completed modules, scores, and identified weak areas with AI-generated "
    "insights and recommendations. Dashboards for instructors/admins with class-level "
    "analytics, engagement, completion rates, average performance by topic, and "
    "individual student insights including risk of falling behind.\n\n"
    "5. Automated Project Planning Support Using AI Agents: Include AI agents that "
    "help students and/or instructors plan projects by breaking down a project into "
    "phases, tasks, and milestones; suggesting timelines, deliverables, and checklists; "
    "and adjusting plans based on progress updates.\n\n"
    "6. General Expectations: The system should be designed as a modern web application, "
    "architected to integrate with one or more LLM/GenAI providers (e.g., OpenAI or "
    "local models), with security, privacy, and role-based access for students, "
    "instructors, and admins."
)

pm_to_system_engineer = (
    "The Requirement Engineer has completed the requirements. "
    "Create the detailed design document based on those requirements. "
    "Also estimate work, effort, and productivity for the design phase using pages."
)

pm_to_software_engineer = (
    "The System Engineer has completed the architecture design. "
    "Create the implementation plan based on the design and requirements. "
    "Also estimate work, effort, and productivity for the development phase using SLOC."
)

pm_to_test_engineer = (
    "The software design and implementation plan are complete. "
    "Create the test plan based on the requirements, design, and implementation. "
    "Also estimate work, effort, and productivity for the testing phase using test cases."
)

pm_to_documentation_engineer = (
    "Testing is complete. "
    "Create the documentation plan based on the requirements, design, and implementation. "
    "Also estimate work, effort, and productivity for the documentation phase using pages."
)

In [15]:
conversation_flow = [
    ("Customer", "Requirement_Engineer", customer_message),
    ("Requirement_Engineer", "System_Engineer", pm_to_system_engineer),
    ("System_Engineer", "Software_Engineer", pm_to_software_engineer),
    ("Software_Engineer", "Test_Engineer", pm_to_test_engineer),
    ("Test_Engineer", "Documentation_Engineer", pm_to_documentation_engineer),
]

## Helper Functions

These functions send prompts to Llama 3 through Ollama and collect responses.

In [10]:
def build_context(agent_name, user_message, conversation_history=None):
    system_prompt = AGENT_SYSTEM_PROMPTS[agent_name]

    history_text = ""
    if conversation_history:
        last_item = conversation_history[-1]
        history_text = (
            f"\n\n[Previous Phase: {last_item['agent']}]\n"
            f"{last_item['response'][:1000]}\n"
        )

    full_prompt = f"""
{system_prompt}

Previous conversation context:
{history_text}

Current task:
{user_message}

Instructions:
- Return only markdown
- Keep the output compact
- Complete all required sections
- Do not skip the final effort calculation section
"""
    return full_prompt.strip()


In [19]:

all_results = []
conversation_history = []

for sender, receiver, message in conversation_flow:
    print(f"\n{sender} -> {receiver}")
    print("-" * 80)

    result = call_agent(receiver, message, conversation_history)

    print(result["response"][:2000])
    print(f"\nTime: {result['time_seconds']} sec | Tokens: {result['tokens']}")

    all_results.append(result)
    conversation_history.append({
        "agent": receiver,
        "response": result["response"]
    })

print("\nExperiment finished:")


Customer -> Requirement_Engineer
--------------------------------------------------------------------------------
## Use Cases

### UC1: Student Registration and Profile Setup

* As a student, I want to register for the LMS and set up my profile, so I can access personalized learning paths and AI tutor support.
* Precondition: Student has a valid email address and password.
* Postcondition: Student profile is created, and AI tutor is initialized.

### UC2: AI Tutor Chatbot

* As a student, I want to engage with the AI tutor chatbot to get help with course-related questions, so I can clarify concepts and get feedback.
* Precondition: Student is logged in and has access to the AI tutor.
* Postcondition: AI tutor provides relevant answers, explanations, and suggestions for next steps.

### UC3: Instructor Course Creation and Management

* As an instructor, I want to create and manage courses, including setting up learning objectives, assessments, and project plans, so I can effectively t

## Experiment summary

In [20]:


output_dir = "Phase1_Experiment_Results_llama3"
os.makedirs(output_dir, exist_ok=True)

for idx, result in enumerate(all_results, start=1):
    filename = f"{idx:02d}_{result['agent']}.md"
    filepath = os.path.join(output_dir, filename)

    with open(filepath, "w", encoding="utf-8") as f:
        f.write(f"# {result['agent']}\n\n")
        f.write(f"**Time:** {result['time_seconds']} sec\n\n")
        f.write(f"**Tokens:** {result['tokens']}\n\n")
        f.write("---\n\n")
        f.write(result["response"])

with open(os.path.join(output_dir, "experiment_output.txt"), "w", encoding="utf-8") as f:
    for result in all_results:
        f.write(f"{result['agent']}\n")
        f.write("=" * 80 + "\n")
        f.write(result["response"])
        f.write("\n\n")

with open(os.path.join(output_dir, "summary.json"), "w", encoding="utf-8") as f:
    json.dump(all_results, f, indent=2)

print(f"Saved outputs in: {output_dir}")

Saved outputs in: Phase1_Experiment_Results_llama3


In [36]:
total_tokens = sum(r["tokens"] for r in all_results)
total_elapsed = sum(r["time_seconds"] for r in all_results)

print("=" * 60)
print("EXPERIMENT COMPLETE")
print("=" * 60)

print(f"  Model:           {MODEL}")
print(f"  Framework:       {FRAMEWORK}")
print(f"  Workflow:        Sequential Waterfall Pipeline")
print(f"  Total Agents:    {len(all_results)}")
print(f"  Total Steps:     {len(all_results)}")
print(f"  Total Tokens:    {total_tokens:,}")
print(f"  Total Time:      {total_elapsed:.1f}s")

print()
print("  Agent Breakdown:")

for r in all_results:
    print(f"    {r['agent']:30s} | Tokens: {r['tokens']:6,} | Response: {len(r['response']):6,} chars")

print()
print(f"  Output Directory: {output_dir}/")
print("=" * 60)

EXPERIMENT COMPLETE
  Model:           llama3
  Framework:       Ollama
  Workflow:        Sequential Waterfall Pipeline
  Total Agents:    5
  Total Steps:     5
  Total Tokens:    1,618
  Total Time:      229.8s

  Agent Breakdown:
    Requirement_Engineer           | Tokens:    366 | Response:  2,355 chars
    System_Engineer                | Tokens:    306 | Response:  2,309 chars
    Software_Engineer              | Tokens:    480 | Response:  2,683 chars
    Test_Engineer                  | Tokens:    304 | Response:  1,991 chars
    Documentation_Engineer         | Tokens:    162 | Response:  1,101 chars

  Output Directory: Phase1_Experiment_Results_llama3/


In [21]:
report_path = os.path.join(output_dir, "Phase1_Waterfall_Complete_Report.md")

with open(report_path, "w", encoding="utf-8") as f:
    f.write("# Phase 1 Waterfall Complete Report\n\n")

    for r in all_results:
        f.write(f"## {r['agent']}\n\n")
        f.write(r["response"])
        f.write("\n\n")

print("Complete report generated.")

Complete report generated.
